
---
Methods Project Part 2
---


In [ ]:
#Download and unzip Dataset from kaggle as uci archive returns 502 bad gateway
#https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
!curl -L -o breast-cancer-wisconsin-data.zip\
  https://www.kaggle.com/api/v1/datasets/download/uciml/breast-cancer-wisconsin-data
!unzip breast-cancer-wisconsin-data.zip

Importing all the libraries and preprocessing the data for future as done in part 1

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import time
import plotly.express as px
import joblib
import warnings
import tensorflow as tf
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer
from sklearn.preprocessing import StandardScaler

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from imblearn.over_sampling import SMOTE

from keras.datasets import mnist
from keras import layers, models
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

# Global Variables for consistency
RANDOM_STATE = 1 
K_FOLD = 5

# Same as part 1

# Load the dataset
bcw_df = pd.read_csv('data.csv')
#Column Id and Unnamed: 32 are not useful, One is just Ids for reference and other appears to only contain NaNs
if 'Unnamed: 32' in bcw_df.columns and 'id' in bcw_df.columns:
    bcw_df.drop(columns=['id','Unnamed: 32'],inplace=True)
#Separate target and features before adding noise to the data
X = bcw_df.drop(columns=['diagnosis'])
y = bcw_df['diagnosis'].map(lambda value: 1 if value == 'M' else 0 )

# Stratified Train test split (80/20) for imbalanced dataset, to ensure that the even distribution of classes.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)


## 1. Hyperparameter optimization
For the hyperparamter optimzation we create a function that will return best f1 score along with other metrics such as average runtime, standard deviation of the f1 score, best set of hyperparamters. We also run this while not fixing randomness to get a better idea of the variance across 5 iterations. 

In [32]:
ITERATIONS = 5
def get_best_optimization_results(model:GridSearchCV | BayesSearchCV):
    total_time = 0
    f1_scores = []
    top_params = None
    avg_time = 0
    best_score = 0
    for _ in range(ITERATIONS):
        start_time = time.time()
        model.fit(X_train,y_train)
        total_time += time.time() - start_time
        f1_scores.append(model.best_score_)
        if model.best_score_ > best_score:
            best_score = model.best_score_
            top_params = model.best_params_
    avg_time = total_time/ITERATIONS
    return avg_time, np.mean(f1_scores),np.std(f1_scores), top_params,best_score

For both Bayesian and Grid Optimization, we define the exact same grid space. Even though a continuous search space is preferred for Bayesian optimization, trying a discrete space search here will give us an apples to apples consistency when comparing the efficiency of the two search methods.

**Parameters that we kept fixed or left as default:**

* `max_leaf_nodes=None` (Default): Redundant, as we are already restricting tree growth via `max_depth` and `min_samples_split`.
* `min_impurity_decrease=0.0` (Default): Not needed as this is effective for large noisy dataset.
* `bootstrap=True` (Default): Bootstrap sampling is preferred way for random forest.
* `oob_score=False` (Default): Not required, since we are explicitly using Stratified K-Fold cross-validation to measure performance.
* `n_jobs=-1`: This will allow jobs to run in parallel across all the processors.
* `verbose=0` (Default): Disables logging output.
* `warm_start=False` (Default): Not applicable, as we are fitting entirely independent models during each iteration of the search.
* `min_weight_fraction_leaf=0.0` (Default): Unnecessary since we are already actively tuning `min_samples_leaf`.
* `class_weight=None` (Default): Left as default.
* `max_samples=None` (Default): As the data set is already small we can leave this.

In [33]:
# Parameter grids for both coarse and fine search
fine_param_grid = {
    'n_estimators': [50,75,100,125,150,200,250,300],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 4,6,8, 10],
    'min_samples_leaf': [1, 2,3, 4,5],
    'max_features': ['sqrt', 'log2', None],
    'ccp_alpha': [0.0, 0.01, 0.02]
}
coarse_param_grid = {
    'n_estimators': [100,200,300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2', None],
    'ccp_alpha': [0.0, 0.02]
}

bayes_fine_grid = {
    'n_estimators': Integer(100, 300),
    'max_depth': Categorical([None, 5, 10, 15, 20, 25]),
    'min_samples_split': Integer(2, 10),
    'min_samples_leaf': Integer(1, 5),
    'max_features': Categorical(['sqrt', 'log2', None]),
    'ccp_alpha': Real(0.0, 0.02)
}

#Even though not required I create a coarse grid for bayes as well
bayes_coarse_grid = {
    'n_estimators': Categorical([100, 200,300]),
    'max_depth': Categorical([None, 10, 20]),
    'min_samples_split': Categorical([5, 10]),
    'min_samples_leaf': Categorical([2, 4]),
    'max_features': Categorical(['sqrt', 'log2', None]),
    'ccp_alpha': Categorical([0.0, 0.02])
}

#Define the scorer for grid search as F1
scorer = make_scorer(f1_score)
#For keep random state to None to capture variance for 5 iterations!
cv = StratifiedKFold(n_splits=K_FOLD, shuffle=True, random_state=None)

random_forest_model = RandomForestClassifier(random_state=None)
grid_search_fine = GridSearchCV(RandomForestClassifier(random_state=None), fine_param_grid, scoring=scorer, cv=cv,n_jobs=-1)
average_time_grid_search_fine, mean_f1_score_grid_search_fine, f1_std_grid_search_fine, top_params_grid_search_fine, best_score_grid_search_fine = get_best_optimization_results(grid_search_fine)

grid_search_coarse = GridSearchCV(RandomForestClassifier(random_state=None), coarse_param_grid, scoring=scorer, cv=cv,n_jobs=-1)
average_time_grid_search_coarse, mean_f1_score_grid_search_coarse, f1_std_grid_search_coarse, top_params_grid_search_coarse, best_score_grid_search_coarse = get_best_optimization_results(grid_search_coarse)
# Limiting Bayes to try 50 combinations in the search space  
bayes_opt_fine = BayesSearchCV(RandomForestClassifier(random_state=None), bayes_fine_grid, scoring=scorer, cv=cv, n_iter=50, random_state=None,n_jobs=-1)
average_time_bayes_opt_fine, mean_f1_score_bayes_opt_fine, f1_std_bayes_opt_fine, top_params_bayes_opt_fine, best_score_bayes_opt_fine = get_best_optimization_results(bayes_opt_fine)

bayes_opt_coarse = BayesSearchCV(RandomForestClassifier(random_state=None), bayes_coarse_grid, scoring=scorer, cv=cv, n_iter=50, random_state=None,n_jobs=-1)
average_time_bayes_opt_coarse, mean_f1_score_bayes_opt_coarse, f1_std_bayes_opt_coarse, top_params_bayes_opt_coarse, best_score_bayes_opt_coarse = get_best_optimization_results(bayes_opt_coarse)

#Saving the top bayes run parameters for future use to avoid reruns
joblib.dump(top_params_bayes_opt_fine, 'top_params_bayes_opt.pkl')

print('---')
print(f'Grid Search (Fine)    |Top Score: {best_score_grid_search_fine:.4f}  | Avg Time: {average_time_grid_search_fine:.2f}s   | Avg.F1 Score: {mean_f1_score_grid_search_fine:.4f} +/- {f1_std_grid_search_fine:.4f}')
print(f'Grid Search (Coarse)  |Top Score: {best_score_grid_search_coarse:.4f}| Avg Time: {average_time_grid_search_coarse:.2f}s | Avg. F1 Score: {mean_f1_score_grid_search_coarse:.4f} +/- {f1_std_grid_search_coarse:.4f}')
print(f'Bayes Search (Fine)   |Top Score: {best_score_bayes_opt_fine:.4f}.   | Avg Time: {average_time_bayes_opt_fine:.2f}s     | Avg. F1 Score: {mean_f1_score_bayes_opt_fine:.4f} +/- {f1_std_bayes_opt_fine:.4f}')
print(f'Bayes Search (Coarse) |Top Score: {best_score_bayes_opt_coarse:.4f}  | Avg Time: {average_time_bayes_opt_coarse:.2f}s   | Avg. F1 Score: {mean_f1_score_bayes_opt_coarse:.4f} +/- {f1_std_bayes_opt_coarse:.4f}')
print('---')



---
Grid Search (Fine)    |Top Score: 0.9674  | Avg Time: 757.58s   | Avg.F1 Score: 0.9639 +/- 0.0034
Grid Search (Coarse)  |Top Score: 0.9605| Avg Time: 23.23s | Avg. F1 Score: 0.9566 +/- 0.0027
Bayes Search (Fine)   |Top Score: 0.9613.   | Avg Time: 25.65s     | Avg. F1 Score: 0.9568 +/- 0.0029
Bayes Search (Coarse) |Top Score: 0.9587  | Avg Time: 23.24s   | Avg. F1 Score: 0.9546 +/- 0.0027
---


* For fine grid, exhaustive grid search took ~ 13 minutes while giving the best score but bayes took significantly less time but delivered a decent F1 score. This shows bayes optimization ability to find near optimal hyperparamters from search space by making educated guesses.
* For the coarses grid, the grid search took less time than bayes. This is because when there are less number of parameters it is better to blindly try all combinations (Grid search) rather pausing after eaach iteration and making a educated next best guess (Bayes)  

In [95]:
fig = px.scatter(
    {
    'Method': ['Grid Search', 'Grid Search', 'Bayesian Opt', 'Bayesian Opt'],
    'Grid Type': ['Fine', 'Coarse', 'Fine', 'Coarse'],
    'Avg_Time': [
        average_time_grid_search_fine, average_time_grid_search_coarse,
        average_time_bayes_opt_fine, average_time_bayes_opt_coarse
    ],
    'F1_Mean': [
        best_score_grid_search_fine, best_score_grid_search_coarse,
        best_score_bayes_opt_fine, best_score_bayes_opt_coarse
    ],
    'F1_Std': [
            f1_std_grid_search_fine, f1_std_grid_search_coarse,
            f1_std_bayes_opt_fine, f1_std_bayes_opt_coarse
    ]
}, 
    x="Avg_Time", 
    y="F1_Mean", 
    color="Method", 
    symbol="Grid Type",
    text="Grid Type",
    title="Hyperparameter Optimization: F1 Score vs. Execution Time",
    labels={
        "Avg_Time": "Average Time Spent (seconds)",
        "F1_Mean": "Best F1 Score"
    },
    error_y="F1_Std",
)
fig.update_traces(marker=dict(size=12), textposition='top center')
fig.update_layout(legend_title_text='Search Method',
width=1000, 
height=1000)
fig.show()

## 2. Data Augmentation

For data augmentation we take random index for minority classes two ways, uniformly and randomly top p index which are hard to predict. To get top p index which are hardest to predict we perform logistic regression on the entire dataset. 

In [ ]:
bayes_opt_rf_params = joblib.load("top_params_bayes_opt.pkl")
p = 0.25

# Stratified Train test split (80/20) for imbalanced dataset, to ensure that the even distribution of classes.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
# Get the indices for majority and minority classes over the entire dataset!
majority_index = y_train[y_train == 0].index
minority_index = y_train[y_train == 1].index

# A) Uniformly at random
# uniformly get p% indices from the minority class!
uniform_minority_index = y_train[y_train == 1].sample(frac=p).index

# B) Based on hardness
# Scaling before logistic regression for converegence and better estimates
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train)
logit_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logit_model.fit(X_scaled_train, y_train)
X_scaled_test = scaler.transform(X_test)
# https://stackoverflow.com/questions/54875765/sklearn-return-top-3-classes-from-logistic-regression
# Get the probabilities estimates for the all samples
logit_probability_estimates = logit_model.predict_proba(X_scaled_train)
logit_minority_probability_estimates = logit_probability_estimates[y_train == 1][:, 1]
# Sort minority class by probability ascending (lowest first = hardest to classify)
hard_minority_idx = minority_index[np.argsort(logit_minority_probability_estimates)]
hard_minority_asc_p_index = hard_minority_idx[:int(len(hard_minority_idx) * p)]

def evaluate_smote(min_idx, k_neighbors=None):
    """Evaluates Random Forest with/without SMOTE over 5 iterations on a single split structure."""
    f1_scores = []
    # We combine the majority class with the selected minority class to create the training subset.
    combined_idx = list(majority_index) + list(min_idx)
    X_subset = X_scaled_train[combined_idx]
    y_subset = y_train[combined_idx]
    

    for i in range(5):
        
        if k_neighbors is not None:
            # We do not fix random state for smote to because so that we can capture the variance.
            smote = SMOTE(k_neighbors=k_neighbors, random_state=None)
            X_train_augmented, y_train_augmented = smote.fit_resample(X_subset, y_subset)
        else:
            X_train_augmented, y_train_augmented = X_subset, y_subset
        # Unpack the best parameters dictionary using **
        random_forest_model = RandomForestClassifier(**bayes_opt_rf_params, random_state=RANDOM_STATE)
        random_forest_model.fit(X_train_augmented, y_train_augmented)
        
        y_pred = random_forest_model.predict(X_scaled_test)
        f1_scores.append(f1_score(y_test, y_pred))
        
    return np.mean(f1_scores),np.std(f1_scores)

uniform_no_smote_f1, uniform_no_smote_std = evaluate_smote(uniform_minority_index, k_neighbors=None)
uniform_k1_smote_f1, uniform_k1_smote_std = evaluate_smote(uniform_minority_index, k_neighbors=1)
uniform_k5_smote_f1, uniform_k5_smote_std = evaluate_smote(uniform_minority_index, k_neighbors=5)

hard_no_smote_f1, hard_no_smote_std = evaluate_smote(hard_minority_asc_p_index, k_neighbors=None)
hard_k1_smote_f1, hard_k1_smote_std = evaluate_smote(hard_minority_asc_p_index, k_neighbors=1)
hard_k5_smote_f1, hard_k5_smote_std = evaluate_smote(hard_minority_asc_p_index, k_neighbors=5)

print('---')
print(f'Uniform Sampling - No SMOTE: F1 Score: {uniform_no_smote_f1:.4f} +/- {uniform_no_smote_std:.4f}')
print(f'Uniform Sampling - SMOTE (k=1): F1 Score: {uniform_k1_smote_f1:.4f} +/- {uniform_k1_smote_std:.4f}')
print(f'Uniform Sampling - SMOTE (k=5): F1 Score: {uniform_k5_smote_f1:.4f} +/- {uniform_k5_smote_std:.4f}')
print(f'Hardness Sampling - No SMOTE: F1 Score: {hard_no_smote_f1:.4f} +/- {hard_no_smote_std:.4f}')
print(f'Hardness Sampling - SMOTE (k=1): F1 Score: {hard_k1_smote_f1:.4f} +/- {hard_k1_smote_std:.4f}')
print(f'Hardness Sampling - SMOTE (k=5): F1 Score: {hard_k5_smote_f1:.4f} +/- {hard_k5_smote_std:.4f}')
print('---')

#Plotly bar charts with error bars
fig = go.Figure()
fig.add_trace(go.Bar(
name='Uniform Sampling',
x=['No SMOTE', 'SMOTE (k=1)', 'SMOTE (k=5)'], y=[uniform_no_smote_f1, uniform_k1_smote_f1, uniform_k5_smote_f1],
error_y=dict(type='data', array=[uniform_no_smote_std, uniform_k1_smote_std, uniform_k5_smote_std])
))
fig.add_trace(go.Bar(
        name='Hardness Sampling',
        x=['No SMOTE', 'SMOTE (k=1)', 'SMOTE (k=5)'], y=[hard_no_smote_f1, hard_k1_smote_f1, hard_k5_smote_f1],
        error_y=dict(type='data', array=[hard_no_smote_std, hard_k1_smote_std, hard_k5_smote_std])
    ))
fig.update_layout(barmode='group',title=f'Random Forest Performance with/without SMOTE (p={p*100}% Minority Sample)',
        xaxis_title='Augmentation Strategy',
        yaxis_title='Average F1 Score (5 iterations on identical split)',
        width=1000, 
        height=600)
fig.show()

---
Uniform Sampling - No SMOTE: F1 Score: 0.9524 +/- 0.0000
Uniform Sampling - SMOTE (k=1): F1 Score: 0.9091 +/- 0.0000
Uniform Sampling - SMOTE (k=5): F1 Score: 0.9019 +/- 0.0308
Hardness Sampling - No SMOTE: F1 Score: 0.7368 +/- 0.0000
Hardness Sampling - SMOTE (k=1): F1 Score: 0.7221 +/- 0.0308
Hardness Sampling - SMOTE (k=5): F1 Score: 0.7994 +/- 0.0361
---


## 3. Transfer Learning

For this part I train two CNNs (2 layer and 3 layer) of different depths on Dataset A, then freeze their convolutional bases and replace and retrain only the two fully-connected classification layers on varying proportions of Dataset B's training data. Performance is measured as averaged F1 on a fixed 20% holdout of Dataset B.

- For `2-Layer CNN` I used two `Conv2D` layers (32 and 64 filters) each followed by `MaxPooling2D` (2×2). Early pooling will help in building low level features.
- For `3-Layer CNN base` I used three `Conv2D` layers (32→64→64 filters, 3×3 kernels) with max-pooling after the first two layers only. The third conv layer operates without pooling to preserve more spatial detail before flattening.
-  `ReLU` has been used as the activation functions throughout the convolutional and first fully-connected layer and `Softmax` on the output layer for 5-class probability estimation.
- For classification layer both the CNN has Two `Dense` layers — 64 units with ReLU, then 5 units with Softmax — matching the 5-class structure of both Dataset A and B (Dataset B labels are remapped from {5–9} to {0–4}).
- Optimizer: `Adam` (default learning rate),Loss: `Categorical crossentropy`, Batch size: 128 for base training on A; 64 for transfer fine-tuning on B (smaller batches help with the reduced dataset sizes at low p%).


In [3]:
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

# We combine the train and test set to a single before masking
X_mnist = np.concatenate([X_train_mnist, X_test_mnist], axis=0)
y_mnist = np.concatenate([y_train_mnist, y_test_mnist], axis=0)

# We reshape the data to be in the format (num_samples, num_features) and normalize pixel values to [0, 1]
image_size = X_train_mnist.shape[1] #28 for MNIST
# (num_samples, 28, 28) -> (num_samples, 28, 28, 1) and normalize pixel values to [0, 1]
X_mnist = X_mnist.reshape(-1, image_size, image_size, 1).astype('float32') / 255

# Dataset A (digits 0-4), Dataset B (digits 5-9)
X_mnist_A,y_mnist_A = X_mnist[y_mnist <= 4], y_mnist[y_mnist <= 4]
X_mnist_B,y_mnist_B = X_mnist[y_mnist > 4], y_mnist[y_mnist > 4] 

# One-hot encoding for the labels
y_mnist_A = to_categorical(y_mnist_A, num_classes=5)
y_mnist_B = to_categorical(y_mnist_B - 5, num_classes=5) # {5..9} -> {0..4} to match the 5-class output head structure.

X_mnist_B_train, X_mnist_B_test, y_mnist_B_train, y_mnist_B_test = train_test_split(X_mnist_B, y_mnist_B, test_size=0.2, random_state=RANDOM_STATE)

In [4]:
def get_2_layer_cnn_base():
    base = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten()
    ], name="base_2_layer_cnn")
    return base

def get_3_layer_cnn_base():
    base = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        # I have choosen for no pooling layer here to preserve remaining spatial detail before flattening
        layers.Flatten()
    ], name="base_3_layer_cnn")
    return base

def build_and_compile_full_model(base_model):
    """Attaches 2 Fully-Connected layers"""
    inputs = layers.Input(shape=(28, 28, 1))
    # Pass the inputs through the base feature extractor
    x = base_model(inputs)
    
    # Classification head with 2 fully connected layers
    x = layers.Dense(64, activation='relu', name="fc_1")(x)
    outputs = layers.Dense(5, activation='softmax', name="fc_2_out")(x)
    
    # Combine into a single model
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

To fit the models on Dataset A I run for up to 100 epochs, but since the task converges relatively quickly, `Early stopping` is applied to halt training once performance stops improving and avoid overfitting.

In [5]:
early_stopping = EarlyStopping(
    monitor='val_loss',         
    patience=3,                 
    restore_best_weights=True,  # Revert to the best model, not the last epoch's model
    verbose=1              
)

In [6]:
print("Training 2-layer CNN on Dataset A")
base_2 = get_2_layer_cnn_base()
model_2_data_A = build_and_compile_full_model(base_2)
model_2_data_A.fit(X_mnist_A, y_mnist_A, epochs=10, batch_size=128, verbose="auto", validation_split=0.1,callbacks=[early_stopping])


Training 2-layer CNN on Dataset A
Epoch 1/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9612 - loss: 0.1385 - val_accuracy: 0.9924 - val_loss: 0.0250
Epoch 2/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9917 - loss: 0.0282 - val_accuracy: 0.9961 - val_loss: 0.0114
Epoch 3/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9950 - loss: 0.0178 - val_accuracy: 0.9969 - val_loss: 0.0085
Epoch 4/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9951 - loss: 0.0153 - val_accuracy: 0.9989 - val_loss: 0.0061
Epoch 5/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9971 - loss: 0.0106 - val_accuracy: 0.9978 - val_loss: 0.0053
Epoch 6/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9975 - loss: 0.0081 - val_accuracy: 0.9994 - val_loss: 0.0035
Epoch 7/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9978 - loss: 0.0068 - val_accuracy: 0.9975 - val_loss: 0.0081
Epoch 8/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9990

In [7]:
print("Training 3-layer CNN on Dataset A")
base_3 = get_3_layer_cnn_base()
model_3_data_A = build_and_compile_full_model(base_3)
model_3_data_A.fit(X_mnist_A, y_mnist_A, epochs=10, batch_size=128, verbose="auto", validation_split=0.1,callbacks=[early_stopping])

Training 3-layer CNN on Dataset A
Epoch 1/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9628 - loss: 0.1379 - val_accuracy: 0.9961 - val_loss: 0.0181
Epoch 2/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9912 - loss: 0.0275 - val_accuracy: 0.9978 - val_loss: 0.0080
Epoch 3/10
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9946 - loss: 0.0174 - val_accuracy: 0.9966 - val_loss: 0.0091
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


Transfer experiments use stratified subsamples of p% of the 80% training pool (p ∈ {5, 10, 25, 50, 75, 100}).

In [8]:
def calculate_f1(model, x_test, y_test):
    """Predicts and returns the average class-wise (macro) F1 score."""
    # Convert softmax predictions to class indices
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    # Convert one-hot encoded true labels back to class indices
    y_true_classes = np.argmax(y_test, axis=1)
    return f1_score(y_true_classes, y_pred, average='macro')

# We freeze the base layers.
base_2.trainable = False
base_3.trainable = False
# Percentages of available Dataset B training data
p_values = [5, 10, 25, 50, 75, 100] 

# Dictionaries to store results
transfer_f1_2_layer = []
transfer_f1_3_layer = []

print("Evaluating Transfer Learning on Dataset B...")
for p in p_values:
    if p == 100:
        x_train_p, y_train_p = X_mnist_B_train, y_mnist_B_train
    else:
        # We use train_test_split just to easily stratify and sample a fraction
        x_train_p, _, y_train_p, _ = train_test_split(
            X_mnist_B_train, y_mnist_B_train, train_size=(p / 100.0), 
            random_state=RANDOM_STATE, stratify=y_mnist_B_train
        )
    
    # Evaluate 2-Layer Transfer
    # We rebuild the full model to randomly initialize the 2 FC layers, keeping the base frozen
    transfer_model_2 = build_and_compile_full_model(base_2) 
    transfer_model_2.fit(x_train_p, y_train_p, epochs=10, batch_size=64, verbose="auto",callbacks=[early_stopping])
    transfer_f1_2_layer.append(calculate_f1(transfer_model_2, X_mnist_B_test, y_mnist_B_test))
    
    # Evaluate 3-Layer Transfer
    transfer_model_3 = build_and_compile_full_model(base_3)
    transfer_model_3.fit(x_train_p, y_train_p, epochs=10, batch_size=64, verbose="auto",callbacks=[early_stopping])
    transfer_f1_3_layer.append(calculate_f1(transfer_model_3, X_mnist_B_test, y_mnist_B_test))

Evaluating Transfer Learning on Dataset B...
Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8380 - loss: 0.4855  
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9686 - loss: 0.1045
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9781 - loss: 0.0692
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9912 - loss: 0.0449
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9927 - loss: 0.0349
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9949 - loss: 0.0292
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9978 - loss: 0.0228
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9993 - loss: 0.0162
Epoch 9/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9993 - loss: 0.0127
Epoch 10/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0111
Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7423 - loss: 0.7762  
Epoch 2/10
22/22 ━━━━━━━━━

In [9]:
# Calculating the baseline benchmarks by training directly on B (100% of train pool)
print("Calculating Baseline Benchmarks (Training directly on Dataset B)...")
benchmark_base_2 = get_2_layer_cnn_base() # New, unfrozen base
benchmark_model_2 = build_and_compile_full_model(benchmark_base_2)
benchmark_model_2.fit(X_mnist_B_train, y_mnist_B_train, epochs=10, batch_size=128, verbose="auto",callbacks=[early_stopping])
benchmark_f1_2_layer = calculate_f1(benchmark_model_2, X_mnist_B_test, y_mnist_B_test)

benchmark_base_3 = get_3_layer_cnn_base() # New, unfrozen base
benchmark_model_3 = build_and_compile_full_model(benchmark_base_3)
benchmark_model_3.fit(X_mnist_B_train, y_mnist_B_train, epochs=10, batch_size=128, verbose="auto",callbacks=[early_stopping])
benchmark_f1_3_layer = calculate_f1(benchmark_model_3, X_mnist_B_test, y_mnist_B_test)


Calculating Baseline Benchmarks (Training directly on Dataset B)...
Epoch 1/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9440 - loss: 0.1951
Epoch 2/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9852 - loss: 0.0476
Epoch 3/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9897 - loss: 0.0334
Epoch 4/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9926 - loss: 0.0241
Epoch 5/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9934 - loss: 0.0223
Epoch 6/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9941 - loss: 0.0181
Epoch 7/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9950 - loss: 0.0160
Epoch 8/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9962 - loss: 0.0123
Epoch 9/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9966 - loss: 0.0096
Epoch 10/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9972 - loss: 0.0081
Epoch 1/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9393 

In [10]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=p_values, y=transfer_f1_2_layer, 
    mode='lines+markers', name='2-Layer CNN Transfer',
    marker=dict(symbol='circle', size=10)
))

fig.add_trace(go.Scatter(
    x=p_values, y=transfer_f1_3_layer, 
    mode='lines+markers', name='3-Layer CNN Transfer',
    marker=dict(symbol='diamond', size=10)
))

# Benchmark horizontal lines
fig.add_hline(y=benchmark_f1_2_layer, line_dash="dash",
              annotation_text="2-Layer Benchmark", annotation_position="bottom right")
fig.add_hline(y=benchmark_f1_3_layer, line_dash="dash",
              annotation_text="3-Layer Benchmark", annotation_position="top right")

fig.update_layout(
    title='Transferability of Learned Representations (Dataset A -> Dataset B)',
    xaxis_title='Percentage (p%) of Target Domain Training Data Used',
    yaxis_title='Average Class-wise F1 Score',
    width=1000, 
    height=1000,
    legend=dict(x=0.02, y=0.02)
)

fig.show()